# A3 Aggregator — Multi-Seed Statistical Analysis
Mounts all per-seed CSVs from the `moe-fl-seeds` dataset and runs the full
statistical test suite. Run this **after** all 4 seed notebooks have finished.

**Required:** Mount Kaggle dataset `moe-fl-seeds` containing:
- `all_benchmarks_combined_seed0.csv`
- `all_benchmarks_combined_seed1.csv`
- `all_benchmarks_combined_seed2.csv`
- `all_benchmarks_combined_seed3.csv`


In [ ]:
# ================================================================
# AGGREGATOR CONFIG — only defines OUT and PREV_SEEDS_DIR
# No sweep is run here; this notebook only runs A3 statistics.
# ================================================================
import os, time
T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
def elapsed(): return f"{(time.time()-T0)/60:.1f}m"

SEEDS         = []  # no sweep
PREV_SEEDS_DIR = '/kaggle/input/moe-fl-seeds'

print(f'OUT          = {OUT}')
print(f'PREV_SEEDS_DIR = {PREV_SEEDS_DIR}')
print(f'Dataset exists: {os.path.exists(PREV_SEEDS_DIR)}')


## Cell 12 (A3): Multi-Seed Statistical Analysis

Reads `all_benchmarks_multiseed.csv` and runs the full statistical suite with proper seed variance:
- **Friedman** rank test (all 11 methods + 10 excl. oracle)
- **Wilcoxon signed-rank** family comparisons + each method vs FedAvg (n = seeds × 9 conditions)
- **Kruskal-Wallis** FL vs ML vs MoE
- **Spearman** AUPRC vs alpha per dataset
- **Bootstrap 95% CIs** using seed variance (true CIs, not alpha-range)
- **Cohen's d / Hedges' g** effect sizes vs FedAvg
- **Mean ± std bar chart** per dataset

Outputs: `a3_statistics_report.txt`, `a3_statistics_summary.csv`, `a3_wilcoxon.csv`, `chart_a3_mean_std.png`


In [ ]:
# ================================================================
# A3 - LOAD ALL SEED CSVs (current session + any previous sessions)
# ================================================================
import glob, pandas as _pd_loader, os as _os_loader

# Collect per-seed CSVs: current session output + any mounted dataset
_seed_csvs = []

# 1. Current session
for _f in sorted(glob.glob(f'{OUT}/all_benchmarks_combined_seed*.csv')):
    _seed_csvs.append(_f)

# 2. Previous sessions mounted as Kaggle dataset (PREV_SEEDS_DIR in config)
if PREV_SEEDS_DIR and _os_loader.path.exists(PREV_SEEDS_DIR):
    prev_files = sorted(glob.glob(f'{PREV_SEEDS_DIR}/all_benchmarks_combined_seed*.csv'))
    _seed_csvs.extend(prev_files)
    print(f"Mounted dataset at {PREV_SEEDS_DIR}: {len(prev_files)} seed CSVs found")

if _seed_csvs:
    _dfs = []
    seen_seeds = set()
    for _f in _seed_csvs:
        _seed_val = int(_f.split('_seed')[-1].replace('.csv', ''))
        if _seed_val in seen_seeds:
            continue
        seen_seeds.add(_seed_val)
        _df = _pd_loader.read_csv(_f)
        _df['seed'] = _seed_val
        _dfs.append(_df)
        print(f"  Loaded {_os_loader.path.basename(_f)}  ({len(_df)} rows, seed={_seed_val})")
    _master_all = _pd_loader.concat(_dfs, ignore_index=True)
    _master_all.to_csv(f'{OUT}/all_benchmarks_multiseed.csv', index=False)
    n_seeds_loaded = _master_all['seed'].nunique()
    print(f"Master CSV: {len(_master_all)} rows, {n_seeds_loaded} seeds -> all_benchmarks_multiseed.csv")
else:
    print("No per-seed CSVs found - using existing combined CSV if present")

# ================================================================
# A3 - MULTI-SEED STATISTICAL ANALYSIS (runs fully on Kaggle)
# ================================================================
import os, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats

OUT_STAT  = f'{OUT}/a3_statistics_report.txt'
OUT_SUM   = f'{OUT}/a3_statistics_summary.csv'
OUT_WIL   = f'{OUT}/a3_wilcoxon.csv'
OUT_CHART = f'{OUT}/chart_a3_mean_std.png'

# ── Load data ────────────────────────────────────────────────────────────────
ms_path = f'{OUT}/all_benchmarks_multiseed.csv'
ss_path = f'{OUT}/all_benchmarks_combined.csv'

if os.path.exists(ms_path):
    df_a3 = pd.read_csv(ms_path)
    print(f"Loaded multi-seed CSV: {len(df_a3)} rows, {df_a3['seed'].nunique()} seeds")
elif os.path.exists(ss_path):
    df_a3 = pd.read_csv(ss_path)
    if 'seed' not in df_a3.columns:
        df_a3['seed'] = 42
    print("WARNING: multi-seed CSV not found - falling back to single seed. Stats will be low-power.")
else:
    raise FileNotFoundError("No benchmark CSV found. Run the main sweep first.")

N_SEEDS_A3  = df_a3['seed'].nunique()
METHODS_A3  = sorted(df_a3['strategy'].unique())
DS_LIST_A3  = sorted(df_a3['dataset'].unique())
ALPHA_LIST_A3 = sorted(df_a3['alpha'].unique())
CONDITIONS_A3 = [(ds, a) for ds in DS_LIST_A3 for a in ALPHA_LIST_A3]

ML_S   = {'xgb', 'lgbm', 'catboost'}
MOE_S  = {'moe_static', 'moe_performance', 'moe_confidence', 'moe_typology_aware'}
FL_S   = {'fedavg', 'fedprox', 'fednova', 'persfl'}
MLAB   = {
    'fedavg':'FedAvg','fedprox':'FedProx','fednova':'FedNova','persfl':'PersFL',
    'xgb':'XGBoost','lgbm':'LightGBM','catboost':'CatBoost',
    'moe_static':'MoE-Static','moe_performance':'MoE-Perf',
    'moe_confidence':'MoE-Conf','moe_typology_aware':'MoE-TypAware',
}

lines = []
def pr(s=''):
    print(s)
    lines.append(str(s))

pr('=' * 78)
pr('A3 - Multi-Seed Statistical Tests')
pr(f"Seeds: {sorted(df_a3['seed'].unique())}   n_conditions={len(CONDITIONS_A3)}   n_seeds={N_SEEDS_A3}")
pr(f"Effective paired blocks for Wilcoxon: {len(CONDITIONS_A3) * N_SEEDS_A3}")
pr('=' * 78)

# ── Helper: build paired vectors across all seed x condition ─────────────────
def paired_vectors(m1, m2, metric='auprc'):
    v1, v2 = [], []
    for ds, a in CONDITIONS_A3:
        for s in df_a3['seed'].unique():
            mask1 = (df_a3.strategy==m1)&(df_a3.dataset==ds)&(df_a3.alpha.round(3)==round(a,3))&(df_a3.seed==s)
            mask2 = (df_a3.strategy==m2)&(df_a3.dataset==ds)&(df_a3.alpha.round(3)==round(a,3))&(df_a3.seed==s)
            r1 = df_a3[mask1]; r2 = df_a3[mask2]
            if len(r1)==1 and len(r2)==1:
                v1.append(float(r1[metric].values[0]))
                v2.append(float(r2[metric].values[0]))
    return np.array(v1), np.array(v2)

def wilcoxon_safe(x, y):
    diff = x - y
    if len(diff) < 5 or np.all(diff == 0): return float('nan'), 1.0
    try:
        w, p = stats.wilcoxon(x, y, zero_method='wilcox')
        return float(w), float(p)
    except Exception:
        return float('nan'), float('nan')

def cliff_delta(x, y):
    n1, n2 = len(x), len(y)
    if n1 == 0 or n2 == 0: return 0.0
    cnt = sum(1 if xi > yj else (-1 if xi < yj else 0) for xi in x for yj in y)
    return cnt / (n1 * n2)

def effect_label(d):
    d = abs(d)
    if d >= 0.474: return 'large'
    if d >= 0.330: return 'medium'
    if d >= 0.147: return 'small'
    return 'negligible'

def family_vecs(fset, best=False, include_oracle=True):
    methods = [m for m in METHODS_A3 if m in fset
               and (include_oracle or m != 'moe_typology_aware')]
    out = []
    for ds, a in CONDITIONS_A3:
        for s in df_a3['seed'].unique():
            vals = []
            for m in methods:
                r = df_a3[(df_a3.strategy==m)&(df_a3.dataset==ds)&
                          (df_a3.alpha.round(3)==round(a,3))&(df_a3.seed==s)]
                if len(r)==1: vals.append(float(r['auprc'].values[0]))
            if vals: out.append(max(vals) if best else np.mean(vals))
    return np.array(out)

# ── 1. Per-cell summary ───────────────────────────────────────────────────────
pr(); pr('-'*78); pr('1) Per-cell AUPRC mean +/- std across seeds'); pr('-'*78)
summ_a3 = (df_a3.groupby(['strategy','dataset','alpha'])['auprc']
             .agg(['mean','std','count']).reset_index()
             .rename(columns={'mean':'auprc_mean','std':'auprc_std','count':'n_seeds'}))
summ_a3['auprc_std'] = summ_a3['auprc_std'].fillna(0)
summ_a3.to_csv(OUT_SUM, index=False)
pr(f"Wrote {os.path.basename(OUT_SUM)}  ({len(summ_a3)} rows)")
piv = summ_a3.pivot_table(index=['dataset','alpha'], columns='strategy',
                          values='auprc_mean').round(4)
pr("\nAUPRC mean pivot (dataset x alpha x strategy):")
pr(piv.to_string())

piv_std = summ_a3.pivot_table(index=['dataset','alpha'], columns='strategy',
                               values='auprc_std').round(4)
pr("\nAUPRC std pivot:")
pr(piv_std.to_string())

# ── 2. Friedman rank test ─────────────────────────────────────────────────────
pr(); pr('-'*78); pr('2) Friedman rank test'); pr('-'*78)

def friedman_block(methods_list, label):
    blocks = []
    for ds, a in CONDITIONS_A3:
        for s in df_a3['seed'].unique():
            row = []
            for m in methods_list:
                r = df_a3[(df_a3.strategy==m)&(df_a3.dataset==ds)&
                          (df_a3.alpha.round(3)==round(a,3))&(df_a3.seed==s)]
                row.append(float(r['auprc'].values[0]) if len(r)==1 else np.nan)
            if not any(np.isnan(row)):
                blocks.append(row)
    blocks = np.array(blocks)
    n_blocks, k = blocks.shape
    stat, p = stats.friedmanchisquare(*[blocks[:,i] for i in range(k)])
    ranks = np.mean(np.apply_along_axis(lambda r: stats.rankdata(-r), 1, blocks), axis=0)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    pr(f"  [{label}]  n_blocks={n_blocks}  k={k}")
    pr(f"    chi2={stat:.4f}  df={k-1}  p={p:.6f}  {sig}")
    pr("    Average ranks (1=best):")
    for m, r in sorted(zip(methods_list, ranks), key=lambda x: x[1]):
        pr(f"      {MLAB.get(m,m):<22s}  {r:.3f}")

friedman_block(METHODS_A3, 'All 11 methods')
friedman_block([m for m in METHODS_A3 if m != 'moe_typology_aware'], '10 methods (no oracle)')

# ── 3. Wilcoxon: family comparisons ──────────────────────────────────────────
pr(); pr('-'*78); pr('3) Wilcoxon signed-rank -- family comparisons'); pr('-'*78)

wil_rows = []
comparisons = [
    ('MoE_mean(no_oracle) vs FL_mean',  family_vecs(MOE_S, False, False), family_vecs(FL_S)),
    ('MoE_mean(no_oracle) vs ML_mean',  family_vecs(MOE_S, False, False), family_vecs(ML_S)),
    ('MoE_mean(all) vs FL_mean',        family_vecs(MOE_S, False, True),  family_vecs(FL_S)),
    ('MoE_mean(all) vs ML_mean',        family_vecs(MOE_S, False, True),  family_vecs(ML_S)),
    ('MoE_best(no_oracle) vs FL_best',  family_vecs(MOE_S, True,  False), family_vecs(FL_S, True)),
    ('MoE_best(no_oracle) vs ML_best',  family_vecs(MOE_S, True,  False), family_vecs(ML_S, True)),
]
for label, x, y in comparisons:
    if len(x) == 0 or len(x) != len(y): continue
    w, p = wilcoxon_safe(x, y)
    cd = cliff_delta(x, y)
    wins = int((x>y).sum()); losses = int((x<y).sum())
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    pr(f"  [{label}]  n={len(x)}  W={w}  p={p:.4f} {sig}  delta={cd:+.2f} ({effect_label(cd)})  +/-={wins}/{losses}")
    wil_rows.append({'test':'family','comparison':label,'n':len(x),'W':w,'p':p,
                     'cliff_delta':round(cd,4),'effect':effect_label(cd)})

# ── 4. Wilcoxon: each method vs FedAvg ───────────────────────────────────────
pr(); pr('-'*78); pr('4) Wilcoxon signed-rank -- each method vs FedAvg'); pr('-'*78)

for m in sorted(METHODS_A3):
    if m == 'fedavg': continue
    x, y = paired_vectors(m, 'fedavg')
    if len(x) < 5: continue
    w, p = wilcoxon_safe(x, y)
    cd = cliff_delta(x, y)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    pr(f"  [{MLAB.get(m,m):<22s} vs FedAvg]  n={len(x)}  W={w}  p={p:.4f} {sig}  delta={cd:+.2f} ({effect_label(cd)})")
    wil_rows.append({'test':'vs_fedavg','comparison':f'{m} vs fedavg','n':len(x),
                     'W':w,'p':p,'cliff_delta':round(cd,4),'effect':effect_label(cd)})

pd.DataFrame(wil_rows).to_csv(OUT_WIL, index=False)
pr(f"\nWrote {os.path.basename(OUT_WIL)}")

# ── 5. Kruskal-Wallis ─────────────────────────────────────────────────────────
pr(); pr('-'*78); pr('5) Kruskal-Wallis -- FL vs ML vs MoE'); pr('-'*78)

for include_oracle, lbl in [(False,'excl. oracle'), (True,'incl. oracle')]:
    groups = {}
    for fname, fset in [('FL',FL_S),('ML',ML_S),('MoE',MOE_S)]:
        ms = [m for m in METHODS_A3 if m in fset and (include_oracle or m!='moe_typology_aware')]
        groups[fname] = df_a3[df_a3.strategy.isin(ms)]['auprc'].values
    h, p = stats.kruskal(*groups.values())
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    pr(f"  [{lbl}]  H={h:.4f}  df=2  p={p:.4f} {sig}")

# ── 6. Spearman: AUPRC vs alpha ───────────────────────────────────────────────
pr(); pr('-'*78); pr('6) Spearman: AUPRC vs alpha per dataset'); pr('-'*78)

for ds in DS_LIST_A3:
    sub = df_a3[df_a3.dataset == ds]
    rho, p = stats.spearmanr(sub['alpha'], sub['auprc'])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    pr(f"  {ds:<6s}  rho={rho:+.3f}  p={p:.2e} {sig}  (n={len(sub)})")

# ── 7. Bootstrap 95% CIs using seed variance ─────────────────────────────────
pr(); pr('-'*78); pr('7) Bootstrap 95% CIs using seed variance (true CIs)'); pr('-'*78)

rng_a3 = np.random.default_rng(0)
N_BOOT = 10_000
for ds in DS_LIST_A3:
    sub = df_a3[df_a3.dataset == ds]
    means = sub[sub.strategy != 'moe_typology_aware'].groupby('strategy')['auprc'].mean()
    best_m = means.idxmax()
    vals = sub[sub.strategy == best_m]['auprc'].values
    boot = rng_a3.choice(vals, size=(N_BOOT, len(vals)), replace=True).mean(axis=1)
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    pr(f"  {ds:<6s}  best={MLAB.get(best_m,best_m):<18s}  mean={vals.mean():.4f}"
       f"  95% CI=[{ci_lo:.4f}, {ci_hi:.4f}]  (n_obs={len(vals)})")

# ── 8. Cohen's d / Hedges' g vs FedAvg ───────────────────────────────────────
pr(); pr('-'*78); pr("8) Cohen's d / Hedges' g vs FedAvg"); pr('-'*78)

for m in sorted(METHODS_A3):
    if m == 'fedavg': continue
    x, y = paired_vectors(m, 'fedavg')
    if len(x) < 2: continue
    diff = x - y
    n = len(diff)
    d = diff.mean() / (diff.std(ddof=1) + 1e-12)
    g = d * (1 - 3 / (4*(n-1) - 1))
    mag = 'large' if abs(d)>=0.8 else 'medium' if abs(d)>=0.5 else 'small' if abs(d)>=0.2 else 'negligible'
    pr(f"  {MLAB.get(m,m):<22s}  d={d:+.3f}  g={g:+.3f}  ({mag})")

# ── 9. Mean +/- std bar chart ─────────────────────────────────────────────────
pr(); pr('-'*78); pr('9) Mean +/- std AUPRC chart'); pr('-'*78)

COLORS_A3 = {'fl':'#3b82f6', 'ml':'#ef4444', 'moe':'#8b5cf6'}
order_a3 = ['fedavg','fedprox','fednova','persfl','xgb','lgbm','catboost',
            'moe_static','moe_performance','moe_confidence','moe_typology_aware']

fig, axes = plt.subplots(1, len(DS_LIST_A3), figsize=(16, 5))
if len(DS_LIST_A3) == 1: axes = [axes]

for ax, ds in zip(axes, DS_LIST_A3):
    sub = (summ_a3[summ_a3.dataset == ds]
           .groupby('strategy')
           .agg(auprc_mean=('auprc_mean','mean'), auprc_std=('auprc_std','mean'))
           .reset_index())
    sub = (sub.set_index('strategy')
              .reindex([m for m in order_a3 if m in sub.strategy.values])
              .reset_index())
    cols = [COLORS_A3['moe'] if m in MOE_S else COLORS_A3['ml'] if m in ML_S else COLORS_A3['fl']
            for m in sub.strategy]
    ax.bar(range(len(sub)), sub.auprc_mean, yerr=sub.auprc_std,
           color=cols, alpha=0.82, capsize=4, error_kw={'linewidth':1.2})
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([MLAB.get(m,m) for m in sub.strategy],
                       rotation=45, ha='right', fontsize=7)
    ax.set_title(f'{ds}', fontweight='bold')
    ax.set_ylabel('AUPRC (mean +/- std)')
    ax.set_ylim(0, min(1.0, sub.auprc_mean.max() * 1.35 + 0.01))
    ax.spines[['top','right']].set_visible(False)

legend_patches = [Patch(color=COLORS_A3['fl'],label='FL'),
                  Patch(color=COLORS_A3['ml'],label='ML'),
                  Patch(color=COLORS_A3['moe'],label='MoE')]
fig.legend(handles=legend_patches, loc='upper right', fontsize=8)
fig.suptitle(f'AUPRC Mean +/- Std across {N_SEEDS_A3} Seeds', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_CHART, dpi=150, bbox_inches='tight')
plt.close()
pr(f"Chart saved: {os.path.basename(OUT_CHART)}")

# ── Write report ──────────────────────────────────────────────────────────────
pr(); pr('='*78); pr('FILES WRITTEN'); pr('='*78)
for f_ in [OUT_STAT, OUT_SUM, OUT_WIL, OUT_CHART]:
    if os.path.exists(f_):
        pr(f"  {os.path.basename(f_)}  ({os.path.getsize(f_):,} bytes)")

with open(OUT_STAT, 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(lines))
print(f"Written: {os.path.basename(OUT_STAT)}")
